<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Neural Networks and Deep Learning — Solutions
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

*Worked answers to the **» Predict**, **» Investigate**, **» Modify** and **» Make** tasks in **06_Neural_Networks_and_Deep_Learning**, in the same order. Run the setup cell first; it rebuilds the table and the split as the session did. The numbers quoted are for the course views and `random_state=123`. Where a task asked for a decision, the answer here is one defensible choice, not the only one.*

## Setup

In [ ]:
%pip install pandas_gbq --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas_gbq
import sklearn

# scikit-learn steps return tables with their column names, not bare arrays
sklearn.set_config(transform_output='pandas')

# The Google Cloud project that pays for your queries. After the course, use your own.
PROJECT = 'bowyerlab-mimic-teaching'

def query(sql, project=PROJECT):
    """Run a SQL query on BigQuery and return the result as a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=project)

# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.patients
df_patients = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.patients`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.admissions
df_admissions = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.admissions`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.icustays
df_icustays = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.icustays`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_vitalsign
df_vitals = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_vitalsign`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.diagnoses_icd
df_diagnoses = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.diagnoses_icd`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_lab
df_labs = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_lab`")

print(len(df_patients), len(df_admissions), len(df_icustays), len(df_vitals), len(df_diagnoses), len(df_labs))
df_stays = df_icustays.merge(df_admissions, on=['subject_id', 'hadm_id'])
df_stays = df_stays.merge(df_patients, on='subject_id')
df_stays = df_stays.merge(df_vitals, on=['subject_id', 'stay_id'])
df_stays = df_stays.merge(df_labs[['stay_id', 'hemoglobin_min', 'hematocrit_min']], on='stay_id', how='left')

df_stays['died'] = df_stays['hospital_expire_flag'].astype(int)
age = df_stays['anchor_age'] + df_stays['admittime'].dt.year - df_stays['anchor_year']
df_stays['age'] = age.astype(int)
time_to_death = df_stays['deathtime'] - df_stays['intime']
df_stays['dead_by_24h'] = time_to_death.dt.total_seconds() < 24 * 60 * 60

# These columns arrive as Decimal objects (BigQuery NUMERIC), which pandas cannot average: make them floats
for column in ['temperature_min', 'temperature_max', 'temperature_mean', 'hemoglobin_min', 'hematocrit_min', 'los']:
    df_stays[column] = df_stays[column].astype(float)

print(df_stays.shape)
# The first ICU stay of each admission. Sorted by stay_id so the split below is the same for everyone
df_model = df_stays.sort_values(['intime', 'stay_id'])
df_model = df_model.drop_duplicates('hadm_id', keep='first')
df_model = df_model.sort_values('stay_id')

grouping = {
    'EW EMER.': 'EMERGENCY', 'DIRECT EMER.': 'EMERGENCY',
    'URGENT': 'URGENT',
    'ELECTIVE': 'ELECTIVE', 'SURGICAL SAME DAY ADMISSION': 'ELECTIVE',
    'OBSERVATION ADMIT': 'OBSERVATION', 'EU OBSERVATION': 'OBSERVATION',
    'DIRECT OBSERVATION': 'OBSERVATION', 'AMBULATORY OBSERVATION': 'OBSERVATION',
}
admission_group = df_model['admission_type'].replace(grouping)
adm_dummies = pd.get_dummies(admission_group, prefix='adm').astype(int)

race_group = df_model['race'].str.replace(r' -.*$', '', regex=True)
race_group = race_group.str.replace(' OR ', '/')
race_group = race_group.replace(['UNKNOWN', 'UNABLE TO OBTAIN', 'PATIENT DECLINED TO ANSWER', 'OTHER'], 'UNKNOWN/OTHER')
race_dummies = pd.get_dummies(race_group, prefix='race').astype(int)

df_model = pd.concat([df_model, adm_dummies, race_dummies], axis=1)
print(df_model.shape)
# Clinical limits are fixed numbers, not learnt from the rows, so they can come before the split
df_model['glucose_max'] = df_model['glucose_max'].mask(df_model['glucose_max'] > 2000)
df_model['heart_rate_max'] = df_model['heart_rate_max'].mask(df_model['heart_rate_max'] > 250)

temperature_gap = df_model['temperature_mean'].isna()
df_model['temperature_missing'] = temperature_gap.astype(int)

is_icd9_diabetes = df_diagnoses['icd_code'].str.startswith('250')
is_icd10_diabetes = df_diagnoses['icd_code'].str.startswith('E10') | df_diagnoses['icd_code'].str.startswith('E11')
df_diabetes = df_diagnoses[is_icd9_diabetes | is_icd10_diabetes]
has_code = df_model['hadm_id'].isin(df_diabetes['hadm_id'])
df_model['diabetes'] = has_code.astype(int)

numeric_features = ['age', 'heart_rate_mean', 'heart_rate_max', 'resp_rate_mean', 'sbp_min', 'glucose_max']
binary_features = ['temperature_missing', 'diabetes'] + list(adm_dummies.columns) + list(race_dummies.columns)
other_columns = ['subject_id', 'anchor_year_group', 'died', 'los', 'dead_by_24h', 'hemoglobin_min', 'hematocrit_min', 'temperature_mean']

# The model's columns: Session 4's, without diabetes, which is only coded after discharge
kept_binary = binary_features.copy()
kept_binary.remove('diabetes')
features = numeric_features + ['temperature_mean'] + kept_binary

df_features = df_model[other_columns + numeric_features + binary_features].copy()
df_features = df_features.dropna(subset=numeric_features)
print(df_features.shape)
print('Stays that ended in death:', round(df_features['died'].mean(), 3))
from sklearn.model_selection import train_test_split

patients = df_features['subject_id'].unique()
train_patients, test_patients = train_test_split(patients, test_size=0.2, random_state=123)

in_train = df_features['subject_id'].isin(train_patients)
in_test = df_features['subject_id'].isin(test_patients)
df_train = df_features[in_train].copy()
df_test = df_features[in_test].copy()

y_train = df_train['died']
y_test = df_test['died']
print(len(df_train), 'training rows,', len(df_test), 'test rows')
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

prepare = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
prepare.fit(df_train[features])

X_train = prepare.transform(df_train[features])
X_test = prepare.transform(df_test[features])

## Five trios, five models

### Investigate: one split, five models
*The numbers here are from reference searches over the same split; the trios' own results replace them in the session.*

The grey dots, logistic regression on 30 different splits, run from **0.765 to 0.815**: a spread of 0.05 from the choice of test patients alone. Four of the five models, **0.770 to 0.811**, sit inside it, so one split cannot rank them. The gap between logistic regression and the network, 0.798 against 0.811, is a quarter of that spread. Only the decision tree, at 0.756, sits below every dot, and its cross-validated AUC, 0.752, agrees: that one difference is believable.

The cross-validated AUCs say the same with a spread attached: 0.789 ± 0.016 for logistic regression against 0.807 ± 0.018 for the network, a gap of about one standard deviation. Compared fold by fold on the same patients (practice Part 3), though, the network wins in every fold, so its small lead is consistent. In the figure, their bars overlap, as do KNN's and the SVM's; only the decision tree's bar sits clear of logistic regression's.

### Every search chose the simplest model it was offered
In the reference searches, logistic regression chose the smallest `C`, the strongest L2 (ridge) penalty, which shrinks every coefficient towards zero (lasso, L1, would instead set some to exactly zero and drop those features); the tree the largest `min_samples_leaf`; KNN the most neighbours, 200; the SVM the smallest `C`, its strongest penalty, and the smoother of its two `gamma` values; and the network the largest `alpha`. Each was at the edge of its grid, so each search should be extended that way before it is trusted. Together they say something about the data: 25 columns of first-day vital signs and admission details carry a modest signal, and the models that do best are the ones that are stopped from chasing noise.

For logistic regression, extending the grid does not help: `C = 0.001` scores 0.785, below 0.01. And across 0.01 to 100 every value scores within 0.002 of the best, against folds that differ by 0.016. The choice of `C` hardly matters here. Both settings are ridge: the grey dots use the default `C = 1`, a weak L2 penalty, and the group's `C = 0.01` is 100 times stronger. The strong penalty makes the model steadier between training sets (across 30 of them, the median spread of a coefficient falls from 0.019 to 0.015, and the rarest race category's from 0.059 to 0.023), but the spread of AUCs, the same for both (standard deviation 0.013 and 0.014), comes from which patients are tested, and no penalty changes that.

## From one neuron to a network

### Investigate: a neuron you have met
**Logistic regression.** Its coefficients, `coef_`, are the weights; its intercept, `intercept_`, is the bias; and the sigmoid is its activation function.

### Investigate: what one neuron can learn
**The patients with a fever.** In the warmest tenth, 16% died, against 6 to 9% in the middle; the line predicts 6% there, and keeps falling. In the coldest tenth, 25% died and the line predicts 17%. Logistic regression learned "colder is worse", which is right on one side only.

No weight and bias can put it right. The sigmoid of a weighted sum of one feature moves one way only: a positive weight makes risk rise with temperature everywhere, a negative weight makes it fall everywhere. A risk that is high at both ends needs a curve that turns, and one neuron cannot turn.

With ten hidden neurons the curve turns near 37 °C, and the test AUC on temperature alone rises from **0.58 to 0.65**. A clinician would have built `abs(temperature - 37)` by hand, the feature engineering of Session 4; the network found the shape itself.

### Investigate: how many hidden neurons?
**Two.** With one hidden neuron the network is no better than logistic regression: one ReLU bend, fed into the sigmoid, still only falls, and the test AUC on temperature alone is 0.58, the same as logistic regression's. Two bends make the U, at 0.65. Ten or a hundred add nothing (0.65 both); a hundred only adds a small kink near 37.3 °C. With about 6,600 training patients and a single feature there is too little freedom to overfit; the next section shows what happens with 25 features.

### Investigate: how many weights?
The first layer has a weight from each of the 25 features to each of the 10 hidden neurons, **250**, and a bias for each hidden neuron, **10**. The output neuron has a weight from each hidden neuron, **10**, and one bias. **271** in all, against logistic regression's 26.

### Predict: the default network
**0.768** on the test rows: worse than logistic regression's 0.80, and worse than the ten-neuron network's 0.797. On the training rows it scores **0.926**. A network with 2,701 weights, learning from 6,738 patients of whom about 750 died, has learned the training patients rather than the pattern.

## How a network learns

### Investigate: watching it learn
**After about 25 to 50 passes.** The loss on the training rows fell for all 411 passes, but the test AUC peaked early and then fell:

| Passes | Test AUC | Training AUC |
| --: | --: | --: |
| 10 | 0.801 | 0.813 |
| 25 | 0.806 | 0.836 |
| 50 | 0.806 | 0.854 |
| 100 | 0.798 | 0.872 |
| 200 | 0.785 | 0.898 |
| 411 | 0.768 | 0.926 |

After about 50 passes, every further fall in the training loss came from fitting the training patients more closely, not from anything that holds for new ones: the network was overfitting, Session 4's Model C reached by training too long. A falling training loss says the search is working, not that the model is improving. The stretch task on early stopping uses exactly this.

### Investigate: too many weights
With `alpha = 0.00001` the large network scores **0.994** on the training rows and **0.700** on the test rows: Session 4's **Model C**, almost perfect on the patients it learned from and the worst model of the day on new ones. With `alpha = 1`, **0.842** and **0.809**: the penalty keeps the weights small, the network cannot memorise, and it does slightly better on the test rows than logistic regression. The network did not need to be smaller; it needed to be held back.

## Independent practice

### Part 1, Modify: size, penalty and luck
| Network | Mean AUC | Spread between folds |
| :-- | --: | --: |
| `(10,)`, `alpha = 0.0001` | 0.791 | 0.024 |
| `(5,)` | 0.795 | 0.017 |
| `(50,)` | 0.768 | 0.030 |
| `(10,)`, `alpha = 0.01` | 0.791 | 0.023 |
| `(10,)`, `alpha = 1` | **0.802** | 0.016 |

`alpha = 1` did best, but its lead of 0.011 over the starting network is smaller than the spread between folds, so on its own it is not convincing: Part 3 shows how to check, fold by fold. The clearer result is the bad one: 50 neurons with little penalty is worse by 0.02, and its folds vary most. The cell below runs all five.

In [ ]:
# Hurts: 50 neurons with little penalty. Helps: alpha = 1, by 0.011, inside the spread between folds, like the rest
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

settings = [((10,), 0.0001), ((5,), 0.0001), ((50,), 0.0001), ((10,), 0.01), ((10,), 1)]
for size, alpha in settings:
    network = MLPClassifier(hidden_layer_sizes=size, alpha=alpha, random_state=0, max_iter=1000)
    pipeline_net = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), network)
    scores = cross_val_score(
        pipeline_net,
        df_train[features],
        df_train['died'],
        cv=GroupKFold(n_splits=5),
        groups=df_train['subject_id'],
        scoring='roc_auc',
    )
    print(size, alpha, ' mean AUC', round(scores.mean(), 3), ' spread', round(scores.std(), 3))

### Part 1, Modify: random starting weights
From **0.791 to 0.805** over `random_state` 0 to 4, with nothing else changed: a range of 0.014. That is the same size as the gap between logistic regression and the trios' network on the shared split (0.013). A network's score is one draw from its starting weights as well as from the split, so a fair comparison repeats both.

In [ ]:
# How far the test AUC moves: 0.791 to 0.805, about the gap between logistic regression and the network
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier

for seed in range(5):
    model_seed = MLPClassifier(hidden_layer_sizes=(10,), random_state=seed, max_iter=1000)
    model_seed.fit(X_train, y_train)
    prob_seed = model_seed.predict_proba(X_test)[:, 1]
    print('random_state', seed, ' test AUC', round(roc_auc_score(y_test, prob_seed), 3))

### Part 2, Make: tune it, then test it once
The blanks are `(50,)`, `0.01`, `df_train['subject_id']` and `df_test[features]`. With three sizes, `(5,)`, `(10,)` and `(50,)`, and three penalties, 0.0001, 0.01 and 1, the search chooses **`(50,)` with `alpha = 1`**: cross-validated AUC **0.807 ± 0.018**. `alpha = 1` is the edge of the grid, so extend it: at `alpha = 3` the best is 0.795, and at 10, 0.788. So 1 is about right. Tested once, the chosen network scores **0.811**.

Over logistic regression? It is ahead by about 0.02, cross-validated and on the test rows: about one standard deviation of the folds, and the same as the spread from random starting weights. Against that, logistic regression has 26 numbers a clinician can read, trains in a fraction of a second, and gives the same answer every time. Part 3 shows the lead is consistent, fold by fold, so it is probably real, but small. For this table I would still keep logistic regression, for its readability; if the gain held up at another hospital, the network's cost could be worth paying.

In [ ]:
# Chosen: (50,), alpha 1. CV AUC 0.807, test AUC 0.811. Over logistic regression: not on a gain this size
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipeline_net = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), MLPClassifier(random_state=0, max_iter=1000))

# 1. The grid: in a pipeline, each setting's name starts with its step's
grid = {
    'mlpclassifier__hidden_layer_sizes': [(5,), (10,), (50,)],
    'mlpclassifier__alpha': [0.0001, 0.01, 1],
}

# 2. The search, by cross-validation on the training rows, one patient per fold
search = GridSearchCV(pipeline_net, grid, cv=GroupKFold(n_splits=5), scoring='roc_auc')
search.fit(df_train[features], df_train['died'], groups=df_train['subject_id'])
print('Chosen:', search.best_params_, ' CV AUC:', round(search.best_score_, 3))

# 3. Once, at the end: the chosen network on the test rows
prob_search = search.predict_proba(df_test[features])[:, 1]
print('Test AUC:', round(roc_auc_score(df_test['died'], prob_search), 3))

### Part 3, Investigate: fold by fold
**Yes, in all five folds.** Logistic regression scores 0.802, 0.795, 0.798, 0.788 and 0.753 (mean 0.787); the network 0.820, 0.822, 0.818, 0.802 and 0.775 (mean 0.807). The lead, about 0.02, is smaller than the spread between folds, but that spread is shared: the fifth fold is the hardest for both. Compared fold by fold, on the same patients, the network is ahead every time, so the lead is consistent. One caution: the network's settings were chosen by a search on these same folds, which flatters it a little.

### Part 3, Make: can logistic regression catch up?
**About 80% of the gap.** With the five distances from normal, logistic regression scores 0.819, 0.817, 0.815, 0.792 and 0.770 (mean **0.803**), against the network's 0.807. Temperature alone takes it to 0.797. The network still wins every fold, narrowly. Why you might still choose logistic regression: 31 coefficients that a clinician can read, each with a clinical meaning, against thousands of weights; and the same answer on every run. The network found the U-shapes itself; a clinician who knows them nearly matches it in a model they can explain.

In [ ]:
# Closed: from 0.787 to 0.803, about 80% of the gap. Still choose it: 31 coefficients a clinician can read
table = df_train.copy()

# Distance from a normal value, in either direction
table['temperature_from_normal'] = (table['temperature_mean'] - 37).abs()
table['heart_rate_mean_from_normal'] = (table['heart_rate_mean'] - 80).abs()

table['sbp_min_from_normal'] = (table['sbp_min'] - 110).abs()

table['resp_rate_mean_from_normal'] = (table['resp_rate_mean'] - 16).abs()

table['glucose_max_from_normal'] = (table['glucose_max'] - 120).abs()

features_plus = features + ['temperature_from_normal', 'heart_rate_mean_from_normal', 'sbp_min_from_normal',
                             'resp_rate_mean_from_normal', 'glucose_max_from_normal']

aucs_plus = fold_aucs(LogisticRegression(), table, features_plus)
print('With the distances:', aucs_plus.round(3), ' mean', round(aucs_plus.mean(), 3))
print('Network:           ', aucs_net.round(3), ' mean', round(aucs_net.mean(), 3))

### Modify (stretch): give the network the same features
The network rises from 0.807 to **0.814**: knowing the shapes helps it too. Feature engineering is not made obsolete by a network; it makes both models better.

### Modify (stretch): stop early
**23 passes** instead of 411, and the test AUC rises from **0.768 to 0.803**. Early stopping watches rows the network does not learn from, so it stops around the point where the table in "Watching it learn" peaked, before the network starts memorising the training patients. It is a different way of holding the network back from `alpha`, and the two can be combined.

In [ ]:
# Passes: 23, against 411. Test AUC: 0.803, against 0.768. It stops before the network memorises the training rows
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier

model_early = MLPClassifier(early_stopping=True, random_state=0, max_iter=1000)
model_early.fit(X_train, y_train)

prob_early = model_early.predict_proba(X_test)[:, 1]
print('Passes:', model_early.n_iter_, ' test AUC', round(roc_auc_score(y_test, prob_early), 3))

### Make (stretch): a network for length of stay
**No.** The default network scores an RMSE of **5.18 days**, worse than predicting the mean, 4.80, and worse than the line, 4.74: with 100 neurons and a long-tailed output it fits the training patients' longest stays. Ten neurons give **4.81**, no better than the mean. Session 5's answer stands: length of stay depends on what happens after the first day, and no model can find in the table what is not in it.

In [ ]:
# Did the network find what the line could not: no. 5.18 days with the default, 4.81 with ten neurons, against 4.74 and 4.80
from sklearn.metrics import root_mean_squared_error
from sklearn.neural_network import MLPRegressor

for size in [(100,), (10,)]:
    model_stay = MLPRegressor(hidden_layer_sizes=size, random_state=0, max_iter=1000)
    model_stay.fit(X_train, df_train['los'])
    predicted_stay = model_stay.predict(X_test)
    print(size, 'RMSE', round(root_mean_squared_error(df_test['los'], predicted_stay), 2), 'days')

## Closing

### The two opening questions
1. **Deploy the network instead?** Not on this evidence. Its gain over logistic regression, about 0.02, is the size of the variation from the split and from the starting weights, and it costs readability and reproducibility. What would change it: the gain holding over repeated splits and seeds, better calibration, or an external validation at another hospital.
2. **What to show a clinician:** for logistic regression, the coefficients themselves, on scaled features, or the contribution of each feature to this patient's score. For a network there are no such numbers to read, so explanations have to be computed after the fact: which features, changed, would change this patient's risk most, as the talk described. Those are approximations of the model, not the model, and they can mislead.

### Investigate: logistic regression or a deep network?
1. **Logistic regression.** A table of a few dozen columns: today's result.
2. **A deep network**, convolutional. Millions of pixels, and the features, the edge of a collapsed lung, cannot be written down as columns.
3. **A deep network**, a transformer, or a language model built on one. Free text has no columns at all.
4. **Logistic regression.** Two columns; and Session 5 showed the table does not hold the answer, so no model will.

## Appendix exercises

### An artificial neuron by hand
The weighted sums are 0.5 × 65 + 1.7 × 1.7 + 0.001 × 8200 − 52 = **−8.41** for A, and 0.5 × 82 + 1.7 × 1.85 + 0.001 × 9800 − 52 = **1.945** for B. ReLU turns the negative sum into 0 and passes the positive one through: outputs **0** and **1.945**. The weights are not on comparable scales: weight in kilograms dominates, and height in metres barely counts, which is why features are scaled before a network sees them.

In [ ]:
w1 = 0.5
w2 = 1.7
w3 = 0.001
bias = -52

def relu(x):
    return max(0, x)

va1 = 65
va2 = 1.7
va3 = 8200

vb1 = 82
vb2 = 1.85
vb3 = 9800

output_a = relu(w1 * va1 + w2 * va2 + w3 * va3 + bias)
output_b = relu(w1 * vb1 + w2 * vb2 + w3 * vb3 + bias)

print('Neuron output for A:', round(output_a, 3))
print('Neuron output for B:', round(output_b, 3))